In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/light-t2m
!tar -xzf /content/drive/MyDrive/light-t2m-backup.tar.gz -C /content/

In [ ]:
!ls -lh /content/light-t2m/visual_datas/gen_joints/
!ls -lh /content/light-t2m/visual_datas/meshes/

In [ ]:
!apt-get update -qq
!apt-get install -y python3.10-venv -qq
!apt-get install -y python3.10-dev -qq

In [ ]:
!python3.10 -m venv /content/light-t2m-env

In [ ]:
!/content/light-t2m-env/bin/python --version

In [ ]:
!/content/light-t2m-env/bin/python -m pip install "numpy==1.23.5"

In [ ]:
!/content/light-t2m-env/bin/python -m pip install Jinja2 typing-extensions --index-url https://pypi.org/simple

In [ ]:
!/content/light-t2m-env/bin/python -m pip install \
torch==2.2.2 \
torchvision==0.17.2 \
torchaudio==2.2.2 \
--index-url https://download.pytorch.org/whl/cu121

In [ ]:
!/content/light-t2m-env/bin/python -c "import torch; print('Torch:', torch.__version__); print('CUDA:', torch.version.cuda); print('CUDA available:', torch.cuda.is_available()); print('ABI:', torch._C._GLIBCXX_USE_CXX11_ABI)"

In [ ]:
!/content/light-t2m-env/bin/python -m pip install -r /content/light-t2m/requirements_fix.txt

In [ ]:
!/content/light-t2m-env/bin/python -m pip install --force-reinstall "numpy==1.23.5"

In [ ]:
!wget -q --show-progress \
"https://github.com/state-spaces/mamba/releases/download/v2.0.3/mamba_ssm-2.0.3+cu122torch2.2cxx11abiFALSE-cp310-cp310-linux_x86_64.whl" \
-O /content/mamba_ssm-2.0.3.whl

In [ ]:
!ls -lh /content/mamba_ssm-2.0.3.whl

In [ ]:
!mv /content/mamba_ssm-2.0.3.whl \
/content/mamba_ssm-2.0.3+cu122torch2.2cxx11abiFALSE-cp310-cp310-linux_x86_64.whl

In [ ]:
!/content/light-t2m-env/bin/python -m pip install \
--no-deps \
"/content/mamba_ssm-2.0.3+cu122torch2.2cxx11abiFALSE-cp310-cp310-linux_x86_64.whl"

In [ ]:
!/content/light-t2m-env/bin/python -c "import torch, mamba_ssm, selective_scan_cuda; print('Torch:', torch.__version__); print('CUDA:', torch.version.cuda); print('CUDA available:', torch.cuda.is_available()); print('Mamba:', mamba_ssm.__version__); print('selective_scan_cuda: OK')"

In [ ]:
!/content/light-t2m-env/bin/python -m pip install \
"bpy==3.4.0" \
--extra-index-url https://download.blender.org/pypi/

In [ ]:
!/content/light-t2m-env/bin/python -m pip install \
imageio \
matplotlib \
smplx \
h5py \
git+https://github.com/mattloper/chumpy \
imageio-ffmpeg

In [ ]:
!MPLBACKEND=Agg /content/light-t2m-env/bin/python -c "import imageio, bpy, matplotlib, smplx, h5py, chumpy; print('Render Python packages OK'); print('bpy:', bpy.app.version_string)"

In [ ]:
!MPLBACKEND=Agg /content/light-t2m-env/bin/python -c "import sys,numpy as np,torch,mamba_ssm,selective_scan_cuda,bpy,imageio,matplotlib,smplx,h5py,chumpy; print('Python:',sys.version.split()[0]); print('NumPy:',np.__version__); print('Torch:',torch.__version__); print('CUDA:',torch.version.cuda); print('CUDA available:',torch.cuda.is_available()); print('Torch ABI:',torch._C._GLIBCXX_USE_CXX11_ABI); print('Mamba:',mamba_ssm.__version__); print('selective_scan_cuda: OK'); print('bpy:',bpy.app.version_string); print('Render packages: OK')"

In [ ]:
%cd /content/light-t2m

!mkdir -p data/HumanML3D/new_joint_vecs

!apt-get -qq update
!apt-get -qq install unrar

In [ ]:
!which unrar

In [ ]:
from pathlib import Path
import shutil
import sys

DRIVE_BENCHMARK = Path(
    "/content/drive/MyDrive/Light-T2M/benchmark_utils.py"
)

LOCAL_BENCHMARK = Path(
    "/content/benchmark_utils.py"
)

if not DRIVE_BENCHMARK.exists():
    raise FileNotFoundError(
        f"Cannot find:\n{DRIVE_BENCHMARK}"
    )

shutil.copy2(
    DRIVE_BENCHMARK,
    LOCAL_BENCHMARK
)

print("Copied benchmark_utils.py to:")
print(LOCAL_BENCHMARK)


if "/content" not in sys.path:
    sys.path.insert(0, "/content")

from benchmark_utils import (
    standardise_motionhiflow,
    render_standard_gif,
)

print("benchmark_utils imported successfully.")

In [ ]:
from pathlib import Path

OUTPUT_ROOT = Path("/content/LightT2M_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Output root:")
print(OUTPUT_ROOT)

In [ ]:
PROMPTS = [
    "A person walks forward.",
    "A person jumps once.",
    "A person walks to the left.",
    "A person walks forward and then makes a right turn.",
    "A person raises the left hand above the head.",
    "A person walks slowly forward.",
    "A person walks quickly forward.",
    "A person jumps once and then turns right.",
    "A person walks forward, jumps once, and then turns right.",
    "A person walks forward while raising the right hand.",
    "A person walks forward and then turns right.",
    "A person walks backward, jumps twice, and then turns right.",
    "The person moves straight ahead before making a right turn."
]

print("Number of prompts:", len(PROMPTS))

for i, prompt in enumerate(PROMPTS, start=1):
    print(i, prompt)

In [ ]:
# @title
from pathlib import Path
import subprocess
import os
import sys
import numpy as np

from IPython.display import display, HTML
import base64


# ============================================================
# Paths
# ============================================================

LIGHT_T2M_DIR = Path("/content/light-t2m")

# Light-T2M always writes the latest generated motion here.
# We must copy it immediately after each generation.
MOTION_FILE = (
    LIGHT_T2M_DIR
    / "visual_datas/gen_joints/gen_motion_0.npy"
)

OUTPUT_ROOT = Path(
    "/content/LightT2M_outputs"
)

PYTHON = "/content/light-t2m-env/bin/python"


# ============================================================
# Import benchmark_utils
# ============================================================

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

from benchmark_utils import (
    standardise_motionhiflow,
    render_standard_gif,
)


# ============================================================
# Find the next output folder number
# ============================================================

def get_next_output_id():

    OUTPUT_ROOT.mkdir(
        parents=True,
        exist_ok=True
    )

    existing_ids = []

    for folder in OUTPUT_ROOT.iterdir():

        if (
            folder.is_dir()
            and folder.name.isdigit()
        ):
            existing_ids.append(
                int(folder.name)
            )

    if not existing_ids:
        return 1

    return max(existing_ids) + 1


# ============================================================
# Generate ONE prompt
# ============================================================

def generate_motion(prompt, output_id):

    output_dir = (
        OUTPUT_ROOT
        / f"{output_id:03d}"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    print("=" * 60)
    print(
        f"Generating prompt "
        f"{output_id:03d}"
    )
    print(f"Prompt: {prompt}")
    print("=" * 60)


    # ========================================================
    # 1. Generate motion with Light-T2M
    # ========================================================

    command = [
        PYTHON,
        "src/sample_motion.py",
        'device="0"',
        "model.guidance_scale=4",
        "model.noise_scheduler.prediction_type=sample",
        f'text="{prompt}"',
        "length=100",
    ]

    # Prevent Colab's matplotlib backend from being
    # inherited by the Light-T2M subprocess.
    env = os.environ.copy()
    env.pop("MPLBACKEND", None)

    result = subprocess.run(
        command,
        cwd=LIGHT_T2M_DIR,
        text=True,
        capture_output=True,
        env=env,
    )


    if result.returncode != 0:

        print(result.stderr)

        raise RuntimeError(
            f"Light-T2M generation failed "
            f"for prompt {output_id:03d}. "
            f"Return code: {result.returncode}"
        )


    # ========================================================
    # 2. Load the newly generated motion
    # ========================================================

    if not MOTION_FILE.exists():

        raise FileNotFoundError(
            f"Generated motion not found:\n"
            f"{MOTION_FILE}"
        )

    motion = np.load(
        MOTION_FILE
    )

    print(
        "Generated motion shape:",
        motion.shape
    )

    if motion.shape != (
        100,
        22,
        3
    ):

        raise ValueError(
            f"Unexpected motion shape: "
            f"{motion.shape}"
        )


    # ========================================================
    # 3. IMPORTANT:
    #    Immediately save RAW motion
    # ========================================================
    #
    # Light-T2M will overwrite gen_motion_0.npy when the
    # next prompt is generated.
    #
    # Therefore this must happen BEFORE generating the next
    # prompt.

    motion_output = (
        output_dir
        / "motion.npy"
    )

    np.save(
        motion_output,
        motion
    )

    print(
        "Saved raw motion:"
    )
    print(
        motion_output
    )


    # ========================================================
    # 4. Save prompt.txt
    # ========================================================

    prompt_output = (
        output_dir
        / "prompt.txt"
    )

    prompt_output.write_text(
        prompt,
        encoding="utf-8"
    )


    # ========================================================
    # 5. Standardise motion
    # ========================================================

    motion_standardised, metadata = (
        standardise_motionhiflow(
            motion,
            mirror_x=True,
        )
    )

    print(
        "Standardised motion shape:",
        motion_standardised.shape
    )


    # ========================================================
    # 6. Render GIF
    # ========================================================

    gif_output = (
        output_dir
        / "motion.gif"
    )

    render_standard_gif(
        motion_standardised,
        gif_output,
        prompt_id=f"{output_id:03d}",
        prompt=prompt,
    )


    # ========================================================
    # 7. Display GIF in Colab
    # ========================================================

    with open(
        gif_output,
        "rb"
    ) as f:

        gif_data = base64.b64encode(
            f.read()
        ).decode()


    display(
        HTML(
            f"""
            <div>
                <h3>
                    Prompt {output_id:03d}
                </h3>

                <p>
                    {prompt}
                </p>

                <img
                    src="data:image/gif;base64,{gif_data}"
                    style="width:500px;"
                >
            </div>
            """
        )
    )


    # ========================================================
    # 8. Finished
    # ========================================================

    print()
    print("Saved:")

    print(
        f"  prompt.txt: "
        f"{prompt_output}"
    )

    print(
        f"  motion.npy: "
        f"{motion_output}"
    )

    print(
        f"  motion.gif: "
        f"{gif_output}"
    )

    print("=" * 60)


    return {
        "id": output_id,
        "prompt": prompt,
        "directory": output_dir,
        "prompt_file": prompt_output,
        "motion_file": motion_output,
        "gif_file": gif_output,
    }


# ============================================================
# Generate ALL prompts in PROMPTS
# ============================================================

def generate_all_prompts(prompts):

    # Allow both:
    # 1. A single string
    # 2. A list of strings

    if isinstance(prompts, str):
        prompts = [prompts]

    if not prompts:
        print("PROMPTS is empty. Nothing to generate.")
        return []

    # Find the first available ID ONCE
    next_id = get_next_output_id()

    print(f"Starting from output folder {next_id:03d}")
    print(f"Number of prompts: {len(prompts)}")
    print()

    results = []

    # Generate sequentially
    for prompt in prompts:

        result = generate_motion(
            prompt,
            output_id=next_id
        )

        results.append(result)

        next_id += 1

    # Simple summary
    print()
    print("=" * 50)
    print("GENERATION FINISHED")
    print("=" * 50)

    for result in results:
        print(
            f"{result['id']:03d} | "
            f"{result['prompt']}"
        )

    return results


    # --------------------------------------------------------
    # Generate sequentially
    # --------------------------------------------------------

    for prompt in prompts:

        result = generate_motion(
            prompt,
            output_id=next_id
        )

        results.append(
            result
        )

        # IMPORTANT:
        # Increment ONLY after this prompt has completely
        # finished and its files have been saved.

        next_id += 1


    # ========================================================
    # Summary
    # ========================================================

    print()
    print("=" * 60)
    print("BATCH GENERATION FINISHED")
    print("=" * 60)

    for result in results:

        print(
            f"{result['id']:03d} | "
            f"{result['prompt']}"
        )

    return results

In [ ]:
results = generate_all_prompts("A person walks forward.")

In [ ]:
# @title
# ============================================================
# Batch T2M Matching Score
# Light-T2M + HumanML3D official T2M evaluator
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import spacy


# ============================================================
# 1. Paths
# ============================================================

BASE_DIR = "/content/light-t2m"

OUTPUT_ROOT = Path(
    "/content/LightT2M_outputs"
)

REF_PATH = (
    "/content/light-t2m/data/HumanML3D/"
    "new_joints/012314.npy"
)

GLOVE_PATH = (
    "/content/light-t2m/deps/glove"
)

EVALUATOR_DIR = (
    "/content/light-t2m/deps/t2m_guo"
)


# ============================================================
# 2. HumanML3D motion processing
# ============================================================

import src.data.humanml.scripts.motion_process as mp

from src.data.humanml.scripts.paramUtil import (
    t2m_raw_offsets,
    t2m_kinematic_chain
)

from src.data.humanml.common.skeleton import Skeleton


# HumanML3D motion_process settings
mp.l_idx1 = 5
mp.l_idx2 = 8
mp.fid_r = [8, 11]
mp.fid_l = [7, 10]
mp.face_joint_indx = [2, 1, 17, 16]
mp.joints_num = 22

mp.n_raw_offsets = torch.from_numpy(
    t2m_raw_offsets
)

mp.kinematic_chain = t2m_kinematic_chain


# ============================================================
# 3. Reference skeleton
# ============================================================

ref_positions = np.load(
    REF_PATH
)

tgt_skel = Skeleton(
    mp.n_raw_offsets,
    mp.kinematic_chain,
    "cpu"
)

mp.tgt_offsets = tgt_skel.get_offsets_joints(
    torch.from_numpy(
        ref_positions[0]
    ).float()
)


# ============================================================
# 4. HumanML3D text preprocessing
# ============================================================

nlp = spacy.load(
    "en_core_web_sm"
)

from src.data.humanml.scripts.word_vectorizer import (
    WordVectorizer
)

w_vectorizer = WordVectorizer(
    GLOVE_PATH,
    "our_vab"
)


def prepare_text(caption):

    doc = nlp(caption)

    word_list = []
    pos_list = []

    for token in doc:

        word = token.text

        # Ignore punctuation etc.
        if not word.isalpha():
            continue

        # HumanML3D official-style lemmatization
        if (
            token.pos_ == "NOUN"
            or token.pos_ == "VERB"
        ) and word.lower() != "left":

            word_list.append(
                token.lemma_.lower()
            )

        else:

            word_list.append(
                word.lower()
            )

        pos_list.append(
            token.pos_
        )

    tokens = ["sos/OTHER"]

    for word, pos in zip(
        word_list,
        pos_list
    ):
        tokens.append(
            f"{word}/{pos}"
        )

    tokens.append(
        "eos/OTHER"
    )

    # HumanML3D text length
    max_text_len = 20

    sent_len = len(tokens)

    if sent_len < max_text_len + 2:

        tokens += [
            "unk/OTHER"
        ] * (
            max_text_len + 2 - sent_len
        )

    else:

        tokens = tokens[
            :max_text_len + 2
        ]

    word_embeddings = []
    pos_one_hots = []

    for token in tokens:

        word_emb, pos_oh = (
            w_vectorizer[token]
        )

        word_embeddings.append(
            word_emb
        )

        pos_one_hots.append(
            pos_oh
        )

    word_embeddings = torch.tensor(
        np.asarray(word_embeddings),
        dtype=torch.float32
    ).unsqueeze(0)

    pos_one_hots = torch.tensor(
        np.asarray(pos_one_hots),
        dtype=torch.float32
    ).unsqueeze(0)

    text_lengths = torch.tensor(
        [sent_len],
        dtype=torch.long
    )

    return (
        word_embeddings,
        pos_one_hots,
        text_lengths
    )


# ============================================================
# 5. Load official evaluator
# ============================================================

from src.models.evaluator.T2M.evaluator import (
    T2MEvaluator
)

evaluator = T2MEvaluator(
    dataset="hml3d",
    deps_dir=EVALUATOR_DIR
)


# ============================================================
# 6. Calculate one motion score
# ============================================================

def calculate_matching_score(
    caption,
    motion_path
):

    # --------------------------------------------------------
    # Load raw motion
    # --------------------------------------------------------

    positions = np.load(
        motion_path
    )

    if positions.ndim != 3:
        raise ValueError(
            f"Unexpected motion shape: "
            f"{positions.shape}"
        )

    # --------------------------------------------------------
    # Convert [T,22,3]
    # to HumanML3D 263-dim representation
    # --------------------------------------------------------

    data, global_positions, local_positions, l_velocity = (
        mp.process_file(
            positions.copy(),
            0.002
        )
    )

    # --------------------------------------------------------
    # Text
    # --------------------------------------------------------

    (
        word_embeddings,
        pos_one_hots,
        text_lengths
    ) = prepare_text(
        caption
    )

    # --------------------------------------------------------
    # Motion tensor
    # --------------------------------------------------------

    motion_tensor = torch.tensor(
        data,
        dtype=torch.float32
    ).unsqueeze(0)

    motion_lengths = torch.tensor(
        [data.shape[0]],
        dtype=torch.long
    )

    # --------------------------------------------------------
    # Embeddings
    # --------------------------------------------------------

    with torch.no_grad():

        text_embedding = (
            evaluator.extract_text_embedding(
                word_embeddings,
                pos_one_hots,
                text_lengths,
                "cpu"
            )
        )

        motion_embedding = (
            evaluator.extract_motion_embedding(
                motion_tensor,
                motion_lengths
            )
        )

    # --------------------------------------------------------
    # Euclidean distance
    # --------------------------------------------------------

    difference = (
        text_embedding -
        motion_embedding
    )

    matching_score = (
        torch.linalg.vector_norm(
            difference,
            ord=2,
            dim=1
        )[0]
        .item()
    )

    return (
        matching_score,
        positions.shape,
        data.shape
    )


# ============================================================
# 7. Find every generated output folder
# ============================================================

folders = sorted(
    [
        p for p in OUTPUT_ROOT.iterdir()
        if p.is_dir() and p.name.isdigit()
    ],
    key=lambda p: int(p.name)
)

print(
    f"Found {len(folders)} output folders."
)

print()

if not folders:
    raise RuntimeError(
        f"No output folders found in:\n{OUTPUT_ROOT}"
    )


# ============================================================
# 8. Calculate all scores
# ============================================================

results = []

for folder in folders:

    prompt_path = (
        folder / "prompt.txt"
    )

    motion_path = (
        folder / "motion.npy"
    )

    # --------------------------------------------------------
    # Check files
    # --------------------------------------------------------

    if not prompt_path.exists():

        print(
            f"[SKIP] {folder.name}: "
            f"prompt.txt not found"
        )

        continue

    if not motion_path.exists():

        print(
            f"[SKIP] {folder.name}: "
            f"motion.npy not found"
        )

        continue

    # --------------------------------------------------------
    # Read prompt
    # --------------------------------------------------------

    caption = prompt_path.read_text(
        encoding="utf-8"
    ).strip()

    print(
        f"Processing {folder.name}: "
        f"{caption}"
    )

    # --------------------------------------------------------
    # Calculate
    # --------------------------------------------------------

    try:

        (
            score,
            original_shape,
            representation_shape
        ) = calculate_matching_score(
            caption,
            motion_path
        )

        results.append({
            "ID": folder.name,
            "Prompt": caption,
            "Motion_File": str(
                motion_path
            ),
            "Frames": original_shape[0],
            "Joints": original_shape[1],
            "Representation_Dim": representation_shape[1],
            "Matching_Score": score,
        })

        print(
            f"  Matching Score: "
            f"{score:.6f}"
        )

    except Exception as e:

        print(
            f"  ERROR: {e}"
        )

        results.append({
            "ID": folder.name,
            "Prompt": caption,
            "Motion_File": str(
                motion_path
            ),
            "Frames": None,
            "Joints": None,
            "Representation_Dim": None,
            "Matching_Score": np.nan,
        })


# ============================================================
# 9. Results table
# ============================================================

results_df = pd.DataFrame(
    results
)

print()
print("=" * 80)
print("T2M MATCHING SCORE RESULTS")
print("=" * 80)

display(
    results_df
)


# ============================================================
# 10. Save CSV
# ============================================================

RESULT_CSV = (
    OUTPUT_ROOT /
    "matching_scores.csv"
)

results_df.to_csv(
    RESULT_CSV,
    index=False,
    encoding="utf-8-sig"
)

print()
print("Results saved to:")
print(RESULT_CSV)